In [1]:
import numpy as np
import pandas as pd

In [ ]:
# ========== BATCH PROCESSING VERSION ==========
import os
import glob
import numpy as np
import pandas as pd

def preprocess_instance(data_file, solution_file, output_dir, 
                       base_k_dist=40, base_k_time=40):
    """
    Preprocess a single VRPTW instance and save outputs to the specified directory.

    This version builds a richer full-edge dataset for a scoring-based GNN:
    - keeps all directed edges (except self-loops)
    - stores stronger VRPTW edge features
    - creates binary labels from the provided optimal solution
    - replaces the useless constant `is_feasible = 1` with meaningful compatibility features
    """
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"Processing: {os.path.basename(data_file)}")
    print(f"Output: {output_dir}")
    print(f"{'='*70}")

    # Load data
    try:
        df = pd.read_csv(data_file)
    except Exception:
        print("  Warning: Standard CSV read failed, trying whitespace separator")
        df = pd.read_csv(data_file, sep=r'\s+')

    # Normalize expected column names
    rename_map = {
        'CUST NO.': 'node_id',
        'XCOORD.': 'x',
        'YCOORD.': 'y',
        'DEMAND': 'demand',
        'READY TIME': 'ready_time',
        'DUE DATE': 'due_date',
        'SERVICE TIME': 'service_time',
    }
    df.rename(columns=rename_map, inplace=True)

    required_cols = ['node_id', 'x', 'y', 'ready_time', 'due_date', 'service_time']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column '{col}' in {data_file}")

    if 'demand' not in df.columns:
        df['demand'] = 0.0

    numeric_cols = ['node_id', 'x', 'y', 'demand', 'ready_time', 'due_date', 'service_time']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=required_cols).reset_index(drop=True)

    # Depot flag
    depot_id = int(df.loc[0, 'node_id'])
    df["is_depot"] = (df["node_id"].astype(int) == depot_id).astype(int)

    # Parse routes
    routes = []
    with open(solution_file, 'r') as f:
        for line in f:
            if line.startswith('Route'):
                route = [int(x) for x in line.split(':', 1)[1].strip().split()]
                routes.append(route)

    print(f"  Nodes: {len(df)}, Routes: {len(routes)}")

    # Convert to arrays for fast feature generation
    node_ids = [int(x) for x in df['node_id'].tolist()]
    N = len(node_ids)

    x = df['x'].to_numpy(dtype=float)
    y = df['y'].to_numpy(dtype=float)
    demand = df['demand'].to_numpy(dtype=float)
    ready = df['ready_time'].to_numpy(dtype=float)
    due = df['due_date'].to_numpy(dtype=float)
    service = df['service_time'].to_numpy(dtype=float)
    is_depot = df['is_depot'].to_numpy(dtype=int)

    # Basic normalization constants
    coord_scale = max(np.ptp(x), np.ptp(y), 1.0)
    max_due = max(float(np.max(due)), 1.0)
    max_demand = max(float(np.max(demand)), 1.0)
    max_service = max(float(np.max(service)), 1.0)

    # Extract used edges from solution
    used_edges = set()
    for route in routes:
        if len(route) > 0:
            used_edges.add((depot_id, route[0]))
            for i in range(len(route) - 1):
                used_edges.add((route[i], route[i+1]))
            used_edges.add((route[-1], depot_id))

    # Build a full directed graph (without self loops)
    edges_list = []
    edge_attr = []
    y_labels = []

    for i in range(N):
        from_node = node_ids[i]

        for j in range(N):
            if i == j:
                continue

            to_node = node_ids[j]

            dx = x[j] - x[i]
            dy = y[j] - y[i]
            distance = float(np.hypot(dx, dy))
            travel_time = distance  # Euclidean travel time assumption

            earliest_departure = ready[i] + service[i]
            earliest_arrival_j = earliest_departure + travel_time
            latest_departure_i = due[i] + service[i]

            # Compatibility / slack features
            is_time_feasible = int(earliest_arrival_j <= due[j])
            arrival_before_ready = max(0.0, ready[j] - earliest_arrival_j)
            tw_slack = due[j] - earliest_arrival_j
            source_window_width = due[i] - ready[i]
            target_window_width = due[j] - ready[j]
            due_gap = due[j] - due[i]
            ready_gap = ready[j] - ready[i]
            service_gap = service[j] - service[i]
            demand_gap = demand[j] - demand[i]

            # Pair/depot structure
            from_is_depot = int(is_depot[i])
            to_is_depot = int(is_depot[j])
            touches_depot = int(from_is_depot or to_is_depot)

            # Added features
            urgency_i = max(0.0, due[i] - ready[i])
            urgency_j = max(0.0, due[j] - ready[j])
            depot_dist_i = float(np.hypot(x[i] - x[0], y[i] - y[0]))
            depot_dist_j = float(np.hypot(x[j] - x[0], y[j] - y[0]))
            tw_overlap = max(0, min(due[i], due[j]) - max(ready[i], ready[j]))

            in_solution = 1 if (from_node, to_node) in used_edges else 0

            edges_list.append([from_node, to_node])
            edge_attr.append([
                from_node,
                to_node,
                distance,
                distance / coord_scale,
                travel_time,
                demand[i],
                demand[j],
                demand_gap,
                demand[i] / max_demand,
                demand[j] / max_demand,
                ready[i],
                ready[j],
                due[i],
                due[j],
                ready_gap,
                due_gap,
                service[i],
                service[j],
                service_gap,
                source_window_width,
                target_window_width,
                earliest_arrival_j,
                latest_departure_i,
                arrival_before_ready,
                tw_slack,
                int(tw_slack >= 0),
                is_time_feasible,
                from_is_depot,
                to_is_depot,
                touches_depot,
                dx,
                dy,
                abs(dx),
                abs(dy),
                earliest_arrival_j / max_due,
                ready[j] / max_due,
                due[j] / max_due,
                urgency_i,
                urgency_j,
                depot_dist_i,
                depot_dist_j,
                tw_overlap,
                in_solution
            ])
            y_labels.append(in_solution)

    edge_columns = [
        'from', 'to',
        'distance', 'distance_norm', 'travel_time',
        'demand_from', 'demand_to', 'demand_gap', 'demand_from_norm', 'demand_to_norm',
        'ready_from', 'ready_to', 'due_from', 'due_to', 'ready_gap', 'due_gap',
        'service_from', 'service_to', 'service_gap',
        'window_width_from', 'window_width_to',
        'earliest_arrival_to', 'latest_departure_from',
        'waiting_time', 'tw_slack', 'has_nonnegative_slack', 'is_time_feasible',
        'from_is_depot', 'to_is_depot', 'touches_depot',
        'dx', 'dy', 'abs_dx', 'abs_dy',
        'arrival_to_norm', 'ready_to_norm', 'due_to_norm',
        'urgency_from', 'urgency_to', 'depot_dist_from', 'depot_dist_to', 'tw_overlap',
        'in_solution'
    ]

    edge_df = pd.DataFrame(edges_list, columns=['from', 'to'])
    edge_features = pd.DataFrame(edge_attr, columns=edge_columns)
    y_label_df = pd.DataFrame({
        'from': [e[0] for e in edges_list],
        'to': [e[1] for e in edges_list],
        'label': y_labels
    })

    # Safety check: feature-label alignment
    assert len(edge_features) == len(y_label_df), "Feature/label length mismatch"
    assert (edge_features['from'].to_numpy() == y_label_df['from'].to_numpy()).all()
    assert (edge_features['to'].to_numpy() == y_label_df['to'].to_numpy()).all()

    # Save outputs
    edge_df.to_csv(os.path.join(output_dir, 'edge_index.csv'), index=False)
    edge_features.to_csv(os.path.join(output_dir, 'edge_features.csv'), index=False)
    y_label_df.to_csv(os.path.join(output_dir, 'y_label.csv'), index=False)
    df.to_csv(os.path.join(output_dir, 'node_features.csv'), index=False)

    # Summary
    coverage = int(y_label_df['label'].sum())
    total_possible_edges = N * (N - 1)
    time_feasible_edges = int(edge_features['is_time_feasible'].sum())
    nonneg_slack_edges = int(edge_features['has_nonnegative_slack'].sum())

    print(f"  Total edges: {len(edge_df):,} / {total_possible_edges:,}")
    print(f"  Solution edges: {coverage}/{len(used_edges)}")
    print(f"  Time-feasible edges: {time_feasible_edges:,} ({time_feasible_edges/len(edge_df)*100:.2f}%)")
    print(f"  Nonnegative-slack edges: {nonneg_slack_edges:,} ({nonneg_slack_edges/len(edge_df)*100:.2f}%)")
    print(f"  ✓ Files saved to {output_dir}")

    return {
        'instance': os.path.basename(data_file),
        'nodes': N,
        'routes': len(routes),
        'edges': len(edge_df),
        'solution_edges': len(used_edges),
        'preserved_edges': coverage,
        'time_feasible_edges': time_feasible_edges
    }

print("Batch preprocessing function defined!")


Batch preprocessing function defined!


In [3]:
# ========== RUN BATCH PROCESSING ==========

# Configuration
data_dir = "./data"  # Directory containing instance files
output_root = "./processed_data"  # Root directory for outputs

# Find all instance files (C101.csv, C102.csv, etc.)
csv_files = sorted(glob.glob(os.path.join(data_dir, "[prPR]*.csv")))

print(f"Found {len(csv_files)} instances to process:")
print("\nChecking for paired solution files:")
print(f"{'Data File':<20} {'Solution File':<30} {'Status':<10}")
print("-" * 65)

# Validate paired files
valid_pairs = []
for csv_file in csv_files:
    # Get base name without extension (e.g., "C101")
    base_name = os.path.splitext(os.path.basename(csv_file))[0]
    
    # Look for corresponding solution file (try multiple patterns)
    # Pattern 1: base_name_solution.txt
    # Pattern 2: base_name_solution .txt (with space before .txt)
    solution_patterns = [
        f"{base_name}_solution.txt",
        f"{base_name}_solution .txt"  # Handle files with space before extension
    ]
    
    solution_file = None
    for pattern in solution_patterns:
        candidate = os.path.join(data_dir, pattern)
        if os.path.exists(candidate):
            solution_file = candidate
            break
    
    data_name = os.path.basename(csv_file)
    
    if solution_file:
        sol_name = os.path.basename(solution_file)
        print(f"{data_name:<20} {sol_name:<30} {'✓ Found':<10}")
        valid_pairs.append((csv_file, solution_file))
    else:
        sol_name = f"{base_name}_solution.txt"
        print(f"{data_name:<20} {sol_name:<30} {'✗ Missing':<10}")

print("-" * 65)
print(f"\nValid pairs: {len(valid_pairs)}/{len(csv_files)}")

if len(valid_pairs) == 0:
    print("\n⚠️  No valid data-solution pairs found!")
    print("Expected structure:")
    print("  data/")
    print("    ├── C101.csv")
    print("    ├── C101_SOLUTION.txt")
    print("    ├── C102.csv")
    print("    ├── C102_SOLUTION.txt")
    print("    ...")
else:
    # Process each instance
    print(f"\nProcessing {len(valid_pairs)} instances...")
    results = []
    
    for idx, (csv_file, solution_file) in enumerate(valid_pairs, start=1):
        # Create output directory name (inst_001, inst_002, ...)
        instance_name = f"inst_{idx:03d}"
        output_dir = os.path.join(output_root, instance_name)
        
        # Process the instance
        result = preprocess_instance(
            data_file=csv_file,
            solution_file=solution_file,
            output_dir=output_dir,
            base_k_dist=40,
            base_k_time=40
        )
        results.append(result)
    
    # Print summary table
    print(f"\n{'='*80}")
    print("BATCH PROCESSING COMPLETE")
    print(f"{'='*80}")
    print(f"\n{'Instance':<15} {'Nodes':<8} {'Routes':<8} {'Edges':<10} {'Sol.Edges':<12} {'Preserved':<12}")
    print("-" * 80)
    for r in results:
        print(f"{r['instance']:<15} {r['nodes']:<8} {r['routes']:<8} {r['edges']:<10} "
              f"{r['solution_edges']:<12} {r['preserved_edges']:<12}")
    print("-" * 80)
    print(f"\nTotal instances processed: {len(results)}")
    print(f"Output directory: {output_root}/")
    print(f"{'='*80}")

Found 11 instances to process:

Checking for paired solution files:
Data File            Solution File                  Status    
-----------------------------------------------------------------
R101.csv             R101_solution.txt              ✓ Found   
R102.csv             R102_solution .txt             ✓ Found   
R103.csv             R103_solution .txt             ✓ Found   
R104.csv             R104_solution .txt             ✓ Found   
R105.csv             R105_solution .txt             ✓ Found   
R106.csv             R106_solution .txt             ✓ Found   
R107.csv             R107_solution .txt             ✓ Found   
R108.csv             R108_solution .txt             ✓ Found   
R109.csv             R109_solution .txt             ✓ Found   
R110.csv             R110_solution .txt             ✓ Found   
R111.csv             R111_solution .txt             ✓ Found   
-----------------------------------------------------------------

Valid pairs: 11/11

Processing 11 instances

In [4]:
# ========== VERIFY OUTPUT STRUCTURE ==========

# Display the directory structure
def show_directory_tree(path, prefix="", max_depth=3, current_depth=0):
    """Display directory tree structure"""
    if current_depth >= max_depth:
        return
    
    try:
        items = sorted(os.listdir(path))
        dirs = [i for i in items if os.path.isdir(os.path.join(path, i))]
        files = [i for i in items if os.path.isfile(os.path.join(path, i))]
        
        # Show directories first
        for i, d in enumerate(dirs):
            is_last_dir = (i == len(dirs) - 1) and len(files) == 0
            print(f"{prefix}{'└── ' if is_last_dir else '├── '}{d}/")
            new_prefix = prefix + ("    " if is_last_dir else "│   ")
            show_directory_tree(os.path.join(path, d), new_prefix, max_depth, current_depth + 1)
        
        # Show files
        for i, f in enumerate(files):
            is_last = i == len(files) - 1
            print(f"{prefix}{'└── ' if is_last else '├── '}{f}")
    except PermissionError:
        pass

if os.path.exists(output_root):
    print(f"\nOutput directory structure:")
    print(f"{output_root}/")
    show_directory_tree(output_root)
else:
    print(f"\n⚠️  Output directory not found: {output_root}")
    print("Run the batch processing cell first!")


Output directory structure:
./processed_data/
├── inst_001/
│   ├── edge_features.csv
│   ├── edge_index.csv
│   ├── node_features.csv
│   └── y_label.csv
├── inst_002/
│   ├── edge_features.csv
│   ├── edge_index.csv
│   ├── node_features.csv
│   └── y_label.csv
├── inst_003/
│   ├── edge_features.csv
│   ├── edge_index.csv
│   ├── node_features.csv
│   └── y_label.csv
├── inst_004/
│   ├── edge_features.csv
│   ├── edge_index.csv
│   ├── node_features.csv
│   └── y_label.csv
├── inst_005/
│   ├── edge_features.csv
│   ├── edge_index.csv
│   ├── node_features.csv
│   └── y_label.csv
├── inst_006/
│   ├── edge_features.csv
│   ├── edge_index.csv
│   ├── node_features.csv
│   └── y_label.csv
├── inst_007/
│   ├── edge_features.csv
│   ├── edge_index.csv
│   ├── node_features.csv
│   └── y_label.csv
├── inst_008/
│   ├── edge_features.csv
│   ├── edge_index.csv
│   ├── node_features.csv
│   └── y_label.csv
├── inst_009/
│   ├── edge_features.csv
│   ├── edge_index.csv
│   ├── node_featu

In [5]:
# ========== DETAILED OVERVIEW OF PROCESSED DATA ==========

import os
import pandas as pd

def analyze_processed_data(output_root="./processed_data"):
    """
    Provide detailed overview of all processed instances built for scoring-based VRPTW learning.
    """
    if not os.path.exists(output_root):
        print(f"⚠️  Output directory not found: {output_root}")
        return

    instances = sorted([
        d for d in os.listdir(output_root)
        if os.path.isdir(os.path.join(output_root, d))
    ])

    if len(instances) == 0:
        print("No processed instances found!")
        return

    print("="*110)
    print(f"PROCESSED DATA OVERVIEW - {len(instances)} Instances")
    print("="*110)

    summary_data = []

    for inst_dir in instances:
        inst_path = os.path.join(output_root, inst_dir)

        try:
            nodes = pd.read_csv(os.path.join(inst_path, 'node_features.csv'))
            edges = pd.read_csv(os.path.join(inst_path, 'edge_index.csv'))
            edge_features = pd.read_csv(os.path.join(inst_path, 'edge_features.csv'))
            labels = pd.read_csv(os.path.join(inst_path, 'y_label.csv'))

            num_nodes = len(nodes)
            num_edges = len(edges)
            num_solution_edges = int(labels['label'].sum())
            pct_solution = (num_solution_edges / num_edges * 100) if num_edges > 0 else 0.0

            num_time_feasible = int(edge_features['is_time_feasible'].sum()) if 'is_time_feasible' in edge_features.columns else 0
            pct_time_feasible = (num_time_feasible / num_edges * 100) if num_edges > 0 else 0.0

            num_nonneg_slack = int(edge_features['has_nonnegative_slack'].sum()) if 'has_nonnegative_slack' in edge_features.columns else 0
            pct_nonneg_slack = (num_nonneg_slack / num_edges * 100) if num_edges > 0 else 0.0

            avg_distance = edge_features['distance'].mean()
            avg_slack = edge_features['tw_slack'].mean() if 'tw_slack' in edge_features.columns else float('nan')

            solution_mask = labels['label'] == 1
            solution_edges = edge_features.loc[solution_mask]
            avg_sol_distance = solution_edges['distance'].mean() if len(solution_edges) > 0 else 0.0
            avg_sol_slack = solution_edges['tw_slack'].mean() if 'tw_slack' in solution_edges.columns and len(solution_edges) > 0 else float('nan')

            summary_data.append({
                'Instance': inst_dir,
                'Nodes': num_nodes,
                'Total_Edges': num_edges,
                'Time_Feasible': num_time_feasible,
                'Time_Feasible_%': f"{pct_time_feasible:.2f}%",
                'Nonneg_Slack': num_nonneg_slack,
                'Nonneg_Slack_%': f"{pct_nonneg_slack:.2f}%",
                'Solution': num_solution_edges,
                'Solution_%': f"{pct_solution:.2f}%",
                'Avg_Dist': f"{avg_distance:.2f}",
                'Avg_Sol_Dist': f"{avg_sol_distance:.2f}",
                'Avg_Slack': f"{avg_slack:.2f}",
                'Avg_Sol_Slack': f"{avg_sol_slack:.2f}"
            })

        except Exception as e:
            print(f"Error processing {inst_dir}: {e}")
            continue

    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        print("\n📊 INSTANCE STATISTICS")
        print("-"*110)
        print(summary_df.to_string(index=False))
        print("-"*110)

        total_nodes = summary_df['Nodes'].sum()
        total_edges = summary_df['Total_Edges'].sum()
        total_time_feasible = summary_df['Time_Feasible'].sum()
        total_nonneg_slack = summary_df['Nonneg_Slack'].sum()
        total_solution_edges = summary_df['Solution'].sum()

        print(f"\n📈 OVERALL STATISTICS")
        print(f"  Total Nodes: {total_nodes:,}")
        print(f"  Total Edges: {total_edges:,}")
        print(f"  Total Time-Feasible Edges: {total_time_feasible:,} ({total_time_feasible/total_edges*100:.2f}%)")
        print(f"  Total Nonnegative-Slack Edges: {total_nonneg_slack:,} ({total_nonneg_slack/total_edges*100:.2f}%)")
        print(f"  Total Solution Edges: {total_solution_edges:,} ({total_solution_edges/total_edges*100:.2f}%)")
        print(f"  Average Nodes per Instance: {total_nodes/len(summary_data):.1f}")
        print(f"  Average Edges per Instance: {total_edges/len(summary_data):,.1f}")

    print(f"\n📋 COMPATIBILITY DETAILS PER INSTANCE")
    print("="*110)
    for inst_dir in instances:
        inst_path = os.path.join(output_root, inst_dir)

        try:
            edge_features = pd.read_csv(os.path.join(inst_path, 'edge_features.csv'))
            labels = pd.read_csv(os.path.join(inst_path, 'y_label.csv'))

            total = len(edge_features)
            time_feasible = int(edge_features['is_time_feasible'].sum())
            nonneg_slack = int(edge_features['has_nonnegative_slack'].sum())
            solution = int(labels['label'].sum())

            tf_and_solution = ((edge_features['is_time_feasible'] == 1) & (labels['label'] == 1)).sum()
            tf_not_solution = ((edge_features['is_time_feasible'] == 1) & (labels['label'] == 0)).sum()
            infeasible_and_solution = ((edge_features['is_time_feasible'] == 0) & (labels['label'] == 1)).sum()
            infeasible_not_solution = ((edge_features['is_time_feasible'] == 0) & (labels['label'] == 0)).sum()

            print(f"\n🔸 {inst_dir}")
            print(f"  Total Edges: {total:,}")
            print(f"  ├─ Time-Feasible: {time_feasible:,} ({time_feasible/total*100:.2f}%)")
            print(f"  │  ├─ In Solution: {tf_and_solution:,}")
            print(f"  │  └─ Not in Solution: {tf_not_solution:,}")
            print(f"  ├─ Time-Infeasible: {total-time_feasible:,} ({(total-time_feasible)/total*100:.2f}%)")
            print(f"  │  ├─ In Solution: {infeasible_and_solution:,}")
            print(f"  │  └─ Not in Solution: {infeasible_not_solution:,}")
            print(f"  ├─ Nonnegative Slack: {nonneg_slack:,} ({nonneg_slack/total*100:.2f}%)")
            print(f"  └─ Solution Edges: {solution:,}")

        except Exception as e:
            print(f"  Error: {e}")

    if instances:
        print(f"\n📁 SAMPLE DATA FROM {instances[0]}")
        print("="*110)

        inst_path = os.path.join(output_root, instances[0])
        nodes = pd.read_csv(os.path.join(inst_path, 'node_features.csv'))
        edge_features = pd.read_csv(os.path.join(inst_path, 'edge_features.csv'))
        labels = pd.read_csv(os.path.join(inst_path, 'y_label.csv'))

        print(f"\n🔹 Node Features (showing first 5 of {len(nodes)} nodes):")
        print(nodes.head().to_string(index=False))

        combined = edge_features.copy()
        combined['label'] = labels['label']

        solution_edges = combined[combined['label'] == 1].copy()
        print(f"\n🔹 Solution Edges (showing first 10 of {len(solution_edges)} solution edges):")
        print(solution_edges.head(10).to_string(index=False))

        time_feasible_non_solution = combined[(combined['is_time_feasible'] == 1) & (combined['label'] == 0)].copy()
        print(f"\n🔹 Time-Feasible Non-Solution Edges (showing first 5 of {len(time_feasible_non_solution)} edges):")
        print(time_feasible_non_solution.head(5).to_string(index=False))

        time_infeasible_edges = combined[combined['is_time_feasible'] == 0].copy()
        if len(time_infeasible_edges) > 0:
            print(f"\n🔹 Time-Infeasible Edges (showing first 5 of {len(time_infeasible_edges)} edges):")
            print(time_infeasible_edges.head(5).to_string(index=False))
        else:
            print(f"\n🔹 Time-Infeasible Edges: None")

        print(f"\n🔹 Edge Classification Summary:")
        print(f"  Total Edges: {len(combined):,}")
        print(f"  ├─ Time-Feasible: {combined['is_time_feasible'].sum():,} ({combined['is_time_feasible'].sum()/len(combined)*100:.2f}%)")
        print(f"  │  ├─ In Solution: {((combined['is_time_feasible']==1) & (combined['label']==1)).sum():,}")
        print(f"  │  └─ Not in Solution: {((combined['is_time_feasible']==1) & (combined['label']==0)).sum():,}")
        print(f"  ├─ Time-Infeasible: {(combined['is_time_feasible']==0).sum():,} ({(combined['is_time_feasible']==0).sum()/len(combined)*100:.2f}%)")
        print(f"  │  ├─ In Solution: {((combined['is_time_feasible']==0) & (combined['label']==1)).sum():,}")
        print(f"  │  └─ Not in Solution: {((combined['is_time_feasible']==0) & (combined['label']==0)).sum():,}")
        print(f"  └─ Nonnegative Slack: {combined['has_nonnegative_slack'].sum():,} ({combined['has_nonnegative_slack'].sum()/len(combined)*100:.2f}%)")

    print("\n" + "="*110)
    print("✅ Overview Complete!")
    print("="*110)

# Run the analysis
analyze_processed_data(output_root)


PROCESSED DATA OVERVIEW - 11 Instances

📊 INSTANCE STATISTICS
--------------------------------------------------------------------------------------------------------------
Instance  Nodes  Total_Edges  Time_Feasible Time_Feasible_%  Nonneg_Slack Nonneg_Slack_%  Solution Solution_% Avg_Dist Avg_Sol_Dist Avg_Slack Avg_Sol_Slack
inst_001    101        10100           3233          32.01%          3233         32.01%       118      1.17%    33.95        27.72    -31.67          9.31
inst_002    101        10100           5801          57.44%          5801         57.44%       116      1.15%    33.95        27.34     15.25         47.22
inst_003    101        10100           7699          76.23%          7699         76.23%       112      1.11%    33.95        26.24     60.39         83.49
inst_004    101        10100           9183          90.92%          9183         90.92%       109      1.08%    33.95        27.65    105.27        120.34
inst_005    101        10100           4253    